# Tutorial 4: Model Checkpoint and Reload with a Manifest

Save a PyTorch model's weights **and** optimizer state to S3 as individual entries, wrap them in a **`Manifest`** that tracks every reference, then destroy all local state and rebuild the entire training checkpoint from that single manifest.

The `Manifest` class handles batched upload, recall, and cleanup of all referenced entries — no manual bookkeeping of `global_id` strings required.

**Prerequisites:** `pip install "laila-core[s3,torch]"` and a `secrets.toml` with your AWS credentials.

In [ ]:
import torch
import torch.nn as nn
import laila
from laila.pool import S3Pool
from laila.policy.central.memory.schema import Manifest

In [ ]:
laila.read_args("./secrets.toml")

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="ckpt_pool",
)

laila.memory.extend(s3_pool, pool_nickname="ckpt")

## Step 1: Define a model and optimizer

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Run a few dummy training steps so the optimizer accumulates state.

In [ ]:
for _ in range(3):
    x = torch.randn(4, 3, 32, 32)
    loss = model(x).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
print("Model trained (dummy steps)")

## Step 2: Wrap every parameter as a LAILA entry

In [ ]:
weight_entries = {}
for name, tensor in model.state_dict().items():
    entry = laila.constant(data=tensor.detach().cpu(), nickname=f"model.{name}")
    weight_entries[name] = entry
    print(f"  {name} -> {entry.global_id}")

optim_entry = laila.constant(
    data=optimizer.state_dict(),
    nickname="optimizer_state",
)

print(f"  optimizer -> {optim_entry.global_id}")

## Step 3: Build the Manifest

A `Manifest` wraps a nested dict of Entry objects. It tracks every reference internally so you can memorize, recall, and clean up all of them with a single call.

In [ ]:
manifest = Manifest(
    data={
        "model_params": weight_entries,
        "optimizer": optim_entry,
    },
    nickname="my_checkpoint",
)

print(f"Manifest global_id:  {manifest.global_id}")
print(f"Manifest is Entry:   {isinstance(manifest, laila.Entry)}")
print(f"Blueprint keys:      {list(manifest.keys())}")
print(f"Total entries:       {sum(1 for _ in manifest)}")

## Step 4: Memorize everything to S3

One call uploads all referenced entries **and** the manifest's own blueprint. All three operations (`memorize`, `remember`, `forget`) return a `GroupFuture`.

In [ ]:
future = manifest.memorize(pool_nickname="ckpt")
future.wait()

print(f"Uploaded {sum(1 for _ in manifest)} entries + manifest blueprint to S3")

## Step 5: Nuke all local state

After this, the only way to get the model back is through LAILA.

In [ ]:
manifest_nickname = "my_checkpoint"

del model, optimizer
del weight_entries, optim_entry, manifest

print("All local state destroyed")

## Step 6: Reload from the manifest alone

Reconstruct the `Manifest` identity from its nickname, recall the stored blueprint from S3, then use `remember()` to fetch every referenced entry.

In [ ]:
# Rebuild manifest identity from the nickname
manifest = Manifest(nickname=manifest_nickname)

# Recall the manifest's blueprint entry from S3
with laila.guarantee:
    ref = laila.remember(manifest.global_id, pool_nickname="ckpt")
blueprint = ref.data[0]

# Reconstruct the manifest with the recalled blueprint
manifest = Manifest(data=blueprint, nickname=manifest_nickname)

print(f"Blueprint keys:    {list(manifest.keys())}")
print(f"Entries to recall: {sum(1 for _ in manifest)}")

# Recall all referenced entries from S3
future = manifest.remember(pool_nickname="ckpt")
recalled_data = future.data

# Build a lookup so we can map global_ids back to data values
data_map = dict(zip(list(manifest), recalled_data))
print(f"All {len(recalled_data)} entries recalled")

In [ ]:
recalled_state_dict = {
    name: data_map[gid]
    for name, gid in manifest.blueprint["model_params"].items()
}

model = SimpleCNN()
model.load_state_dict(recalled_state_dict)
model.eval()
print("Model reconstructed from S3")

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(data_map[manifest.blueprint["optimizer"]])
print("Optimizer reconstructed from S3")

## Step 7: Verify

In [ ]:
test_input = torch.randn(1, 3, 32, 32)
output = model(test_input)
print(f"Output shape: {output.shape}")
print(f"Output: {output}")

## Clean up

In [ ]:
manifest.forget(pool_nickname="ckpt").wait()
print("All entries + manifest blueprint cleaned up from S3")

## Summary

- Each model parameter and the optimizer state become individual LAILA entries
- A **`Manifest`** is a special Entry (scope `MANIFEST`) whose `.data` is the blueprint dict
- **`manifest.memorize()`** uploads every referenced entry + the manifest itself, returns a `GroupFuture`
- **`manifest.remember()`** recalls every referenced entry, returns a `GroupFuture`
- **`manifest.forget()`** deletes all referenced entries + the manifest itself, returns a `GroupFuture`
- **`manifest.data`** live-fetches all entries from the alpha pool and blocks until every leaf is gathered (synchronous, no caching)
- This pattern scales to any checkpoint size — add schedulers, metadata, or dataset fingerprints as needed